# EDA · Kenya · low birth weight

Dataset: `data/processed/ke_lbw.csv`.
Outcome: low birth weight (LBW), derived from `birth_weight_g < 2500`.

Weighed live births only. This skews toward facility births (97.3% of rows), so it is a limitation on how far the results generalise.

What this notebook covers:
- shape and dtypes; confirm the row count matches the proposal
- missingness (blanks are in `preceding_interval_m` and in a handful of ANC fields; `told_high_bp` is complete in this file)
- the outcome rate
- distributions of the predictors (maternal_age, prior_pregnancies, prior_live_births, prior_stillbirth, prior_loss, preceding_interval_m, multiple, education, wealth_quintile, residence, region, told_high_bp) and the outcome rate within each category

Clear outputs before committing.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA = Path("../../../data/processed/ke_lbw.csv")
if not DATA.exists():            # fall back to a copy next to the notebook
    DATA = Path("ke_lbw.csv")
RANDOM_STATE = 42
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_columns", 60)

# 1. Load and confirm data

In [ ]:
# Load data

df = pd.read_csv(DATA)
print(df.shape, "| country:", df["country"].unique())

- The dataset has 4299 rows and 58 columns
- The dataset is fully Kenyan as it is the only value in the country column 

In [ ]:
# Confirm the rows, event count, Presence of duplicates and country

ER, EE = 4299, 356
assert len(df) == ER, f"Expected {ER}, got {len(df)}"

assert df["low_birth_weight"].sum() == EE, \
    "LBW count differs from the cohort build"

assert not df.duplicated(["caseid", "pidx"]).any(), \
    "Duplicate pregnancies"

assert set(df["country"]) == {"Kenya"}, \
    "File has more countries"

print("Row count, event count, uniqueness, country confirmed.")

In [ ]:
# Check for number of women in the file and also women with more than one row in the dataset and row numbers of children who are twins
print("Women in file:", df["caseid"].nunique(),
      " -Women with more than one row:", (df["caseid"].value_counts() > 1).sum())
print("Twin rows (multiple==1):", (df["multiple"] == 1).sum())


- Women in file: 4299 
- Women with more than one row: 0
- Twin rows: 79


# 2. Sorting Columns 
- **Outcome:** `low_birth_weight`.
- **Base predictors:** known at booking.
- **ANC predictors:** collected during antenatal care.
- **Exploratory only:** `region` and  `told_high_bp`
- **Survey design:** `weight` for rates, `v001` for cluster (with country it
  forms `cluster_id`).
- **Leakage:** never a predictor. For LBW, `birth_weight_g` (defines the
  label), `pregnancy_months` (prematurity is the leading cause of LBW),
  plus every other birth-time column.


In [ ]:
OUTCOME = "low_birth_weight"

BASE_PREDICTORS = [
    "maternal_age", "prior_pregnancies", "prior_live_births",
    "prior_stillbirth", "prior_loss", "preceding_interval_m",
    "multiple", "education", "wealth_quintile", "residence",
]
ANC_PREDICTORS = [
    "anc_visits", "first_anc_month", "bp_measured",
    "urine_tested", "blood_tested", "iron_tablets",
]
EXPLORATORY = ["region", "told_high_bp", "told_high_bp_label"]

DESIGN = ["weight", "v001"]

LEAKAGE = [
    "birth_weight_g",        
    "pregnancy_months",       
    "delivery_place", "baby_sex", "pregnancy_outcome",
    "stillbirth", "early_neonatal_death", "perinatal_death",  
]

# 3. Missing data and value ranges
- Check for missing values in the required columns, then the observed ranges

In [ ]:
cols = BASE_PREDICTORS + ANC_PREDICTORS + EXPLORATORY
missing = pd.DataFrame({
    "missing_n": df[cols].isna().sum(),
    "missing_%": df[cols].isna().mean() * 100,
}).sort_values("missing_%", ascending=False)
missing

In [ ]:
num = [c for c in cols + ["birth_weight_g", "pregnancy_months"]
       if pd.api.types.is_numeric_dtype(df[c])]
display(df[num].agg(["min", "median", "max"]).T)
for c in ["education", "wealth_quintile", "residence"]:
    print(c, "->", df[c].value_counts().to_dict())

**Result:**
- `preceding_interval_m` is the only column with real missingness: 1,197 rows (27.8%). All 1,179 first pregnancies are blank (no previous birth), plus 18 later pregnancies (see §4).
- Six ANC fields have 1-4 blanks each (at most 0.09%); `told_high_bp`, `region` and every base predictor other than the interval are complete. These few ANC rows can be dropped or imputed in the pipeline with negligible effect.
- Ranges are plausible: maternal age 13-46, prior live births 0-13, birth weight 600-6,000 g, first ANC month 0-9 (0 = no ANC), ANC visits 0-20.

# 4. Missingness and outcome sanity

In [ ]:
# 1) Outcome and its source must be consistent (leakage sanity)
print("birth_weight_g missing:", df["birth_weight_g"].isna().sum())
print("low_birth_weight missing:", df["low_birth_weight"].isna().sum())
print("LBW defined where weight missing:",
      df.loc[df["birth_weight_g"].isna(), "low_birth_weight"].notna().sum())

# 2) Completeness of told_high_bp in this file (it was a subsample question in
#    some surveys, so check rather than assume)
print("told_high_bp answered by:", f"{df['told_high_bp'].notna().mean():.0%}", "of women")

# 3) Are stillbirths or non-live outcomes present? (cohort is weighed live births)
print("stillbirth flag counts:", df["stillbirth"].value_counts().to_dict())
print("pregnancy_outcome values:", df["pregnancy_outcome"].value_counts().to_dict())

# 4) Where were the babies born? (weighed births skew to facilities)
place = df["delivery_place"].value_counts()
print("\nDelivery place:\n", place.to_string())
print(f"Facility births: {place.drop(['Home', 'Other'], errors='ignore').sum() / len(df):.1%}")

- `birth_weight_g` and `low_birth_weight` have no missing values, and LBW is never defined where weight is missing.
- `told_high_bp` is answered by 100% of women in this file, so there is no subsample issue here (worth confirming against the DHS questionnaire notes).
- Every row has `stillbirth == 0` and `pregnancy_outcome == "Born alive"`, so the cohort holds live births only; no stillbirth decision is needed.
- Delivery place: 97.3% facility (public 3,271; private 644; NGO/faith-based 267) and 104 home + 13 other births. This confirms the facility-birth skew.

In [ ]:
first = df["prior_pregnancies"] == 0
print("Missing interval among first pregnancies:",
      df.loc[first, "preceding_interval_m"].isna().sum(), "of", first.sum())
print("Missing interval among later pregnancies:",
      df.loc[~first, "preceding_interval_m"].isna().sum())

- Missing interval among first pregnancies: 1179 of 1179
- Missing interval among later pregnancies: 18

## 5. Helper functions

- `weighted_rate(d)`: survey-weighted rate per 1,000.
- `rate_ci(d)`: the weighted rate with an approximate 95% CI using the
  Kish effective sample size. Ignores clustering, so it is slightly too
  narrow; the modelling notebooks use proper clustered intervals.
- `rate_by(d, var)`: per-level table with `n`, `events`, weighted rate,
  CI, and a `few_events` flag for groups with fewer than 30 events.

In [ ]:
def weighted_rate(d, outcome=OUTCOME, per=1000):
    """Survey-weighted rate per 1,000 (or per 100 with per=100)."""
    return per * np.average(d[outcome], weights=d["weight"])


def rate_ci(d, outcome=OUTCOME, per=1000):
    """Weighted rate with approximate 95% CI (Wilson + Kish n_eff)."""
    w, y = d["weight"], d[outcome]
    p = np.average(y, weights=w)
    n_eff = w.sum() ** 2 / (w ** 2).sum()
    z = 1.96
    centre = (p + z ** 2 / (2 * n_eff)) / (1 + z ** 2 / n_eff)
    half = z * np.sqrt(p * (1 - p) / n_eff + z ** 2 / (4 * n_eff ** 2)) / (1 + z ** 2 / n_eff)
    return per * p, per * max(centre - half, 0), per * (centre + half)


def rate_by(d, var, outcome=OUTCOME, order=None):
    rows = []
    for level, g in d.dropna(subset=[var]).groupby(var, observed=True):
        rate, lo, hi = rate_ci(g, outcome)
        rows.append({
            "level": level,
            "n": len(g),
            "events": int(g[outcome].sum()),
            "rate_per_1000": rate,
            "ci_low": lo,
            "ci_high": hi,
            "few_events": int(g[outcome].sum()) < 30,
        })
    out = pd.DataFrame(rows).set_index("level")
    return out.reindex(order) if order is not None else out


def screening_summary(flag, y):
    tp = int(((flag == 1) & (y == 1)).sum())
    fp = int(((flag == 1) & (y == 0)).sum())
    fn = int(((flag == 0) & (y == 1)).sum())
    tn = int(((flag == 0) & (y == 0)).sum())
    return pd.Series({
        "flagged_%":    100 * flag.mean(),
        "sensitivity_%": 100 * tp / max(tp + fn, 1),
        "specificity_%": 100 * tn / max(tn + fp, 1),
        "PPV_%":         100 * tp / max(tp + fp, 1),
        "NPV_%":         100 * tn / max(tn + fn, 1),
    })


def plot_rates(tables, title, ncols=3):
    n = len(tables)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.6 * nrows))
    axes = np.atleast_1d(axes).ravel()
    overall = weighted_rate(df)
    for ax, (var, t) in zip(axes, tables.items()):
        t = t.dropna(subset=["rate_per_1000"])
        x = np.arange(len(t))
        err = [t["rate_per_1000"] - t["ci_low"], t["ci_high"] - t["rate_per_1000"]]
        colours = ["lightgrey" if few else "steelblue" for few in t["few_events"]]
        ax.bar(x, t["rate_per_1000"], yerr=err, capsize=3, color=colours)
        ax.axhline(overall, ls="--", color="black", lw=1)
        ax.set_xticks(x, [str(i) for i in t.index], rotation=30)
        ax.set_title(var)
        ax.set_ylabel("LBW per 1,000")
    for ax in axes[n:]:
        ax.set_visible(False)
    fig.suptitle(title + "\n(dashed = overall rate; grey = <30 events; bars = 95% CI)")
    plt.tight_layout()
    plt.show()

## 6. The outcome

LBW is **not** a rare outcome (unlike perinatal death), so a trivial "not
LBW for everyone" model would score 92% accuracy and is useless — PR-AUC and
balanced class weights are still the right choices, but here they matter
mainly for *identification*, not for avoiding accuracy trap.

In [ ]:
overall = pd.DataFrame({
    "events": [int(df[OUTCOME].sum())],
    "unweighted_per_1000": [1000 * df[OUTCOME].mean()],
    "weighted_per_1000":   [weighted_rate(df)],
}, index=["low_birth_weight"])
display(overall)

# Sanity check: does birth_weight_g < 2500 match low_birth_weight?
mismatch = ((df["birth_weight_g"] < 2500) != df[OUTCOME].astype(bool)).sum()
print("Rows where (birth_weight_g < 2500) differs from low_birth_weight:", int(mismatch))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.8))
for v, label in [(0, "Not LBW"), (1, "LBW")]:
    ax.hist(df.loc[df[OUTCOME] == v, "birth_weight_g"].dropna(),
            bins=40, alpha=0.5, density=True, label=label)
ax.axvline(2500, ls="--", color="black", label="2500 g cut-off")
ax.set_xlabel("Birth weight (g)")
ax.set_title("Birth weight by LBW status (Kenya); unweighted")
ax.legend()
plt.show()

**Result:**
- 356 LBW births among 4,299: **82.8 per 1,000 unweighted (8.3%)** and **78.4 per 1,000 weighted (7.8%)**. The gap is 0.4 percentage points, under the 1-point threshold, so the Kenya weights move the rate very little here.
- `birth_weight_g < 2500` matches `low_birth_weight` in every row (0 mismatches).
- The histogram shows a clean break at 2,500 g because the label is derived directly from weight. The noise is in the weights themselves: 99% are recorded as multiples of 100 g and 216 babies sit at exactly 2,500 g (all counted as not LBW), so babies just either side of the cut-off are separated by rounding.

## 7. Derived variables

We create the groupings clinicians think in, matching the cut-offs in
`dataset.py` so the EDA and the modelling pipeline use the same bins.

In [ ]:
df["first_pregnancy"] = (df["prior_pregnancies"] == 0).astype(int)
first = df["first_pregnancy"] == 1
unknown = (~first) & df["preceding_interval_m"].isna()

# Imputation value = median of the observed intervals in the whole cohort.
# It does not use the outcome. In the modelling notebooks, recompute it on the
# training split only.
IMPUTE_INTERVAL_M = df["preceding_interval_m"].median()
assert IMPUTE_INTERVAL_M == 39, IMPUTE_INTERVAL_M
print(f"Later pregnancies with missing interval: {int(unknown.sum())} "
      f"- imputed to {IMPUTE_INTERVAL_M:.0f} months (cohort median of observed intervals)")
df["interval_imputed"] = unknown.astype(int)
df.loc[unknown, "preceding_interval_m"] = IMPUTE_INTERVAL_M

df["short_interval"] = (df["preceding_interval_m"] < 24).astype(int)

df["age_group"] = pd.cut(
    df["maternal_age"], [0, 17, 24, 34, 60],
    labels=["<18", "18-24", "25-34", "35+"],
)
df["parity_group"] = pd.cut(
    df["prior_live_births"], [-1, 0, 2, 4, 30],
    labels=["0", "1-2", "3-4", "5+"],
)
df["interval_group"] = np.select(
    [first, df["preceding_interval_m"] < 24, df["preceding_interval_m"] < 36],
    ["First pregnancy", "<24 months", "24-35 months"],
    default="36+ months",
)

df["no_anc"]   = (df["anc_visits"] == 0).astype(int)
df["anc4plus"] = (df["anc_visits"] >= 4).astype(int)
df["anc_group"] = pd.cut(
    df["anc_visits"], [-1, 0, 3, 7, 40],
    labels=["None", "1-3", "4-7", "8+"],
)
df["first_anc_trimester"] = pd.cut(
    df["first_anc_month"], [-1, 0, 3, 6, 10],
    labels=["No ANC", "1st (0-3 m)", "2nd (4-6 m)", "3rd (7+ m)"],
)

df["checklist_flag"] = (
    ((df["maternal_age"] < 18) | (df["maternal_age"] >= 35))
    | (df["prior_live_births"] >= 5)
    | (df["preceding_interval_m"] < 24)
    | (df["multiple"] == 1)
    | (df["prior_stillbirth"] == 1)
).astype(int)

print("\nFlags and coverage (%):")
display(
    df[["first_pregnancy", "short_interval", "no_anc",
        "anc4plus", "checklist_flag"]].mean().mul(100).round(1)
)

In [ ]:
print("Share of twin pregnancies that are LBW:",
      f"{df.loc[df['multiple']==1, 'low_birth_weight'].mean():.1%}")
print("Share of single pregnancies that are LBW:",
      f"{df.loc[df['multiple']==0, 'low_birth_weight'].mean():.1%}")

- Share of twin pregnancies that are LBW: 49.4%
- Share of single pregnancies that are LBW: 7.5%

**Result:**
- Overall LBW is 8.3% unweighted (about 1 in 12).
- Of 4,299 pregnancies, 27.4% are first pregnancies, 15.7% have a preceding interval under 24 months, 0.9% had no ANC, 67.8% had 4+ ANC visits and **36.4%** are flagged by the checklist.
- Twin pregnancies are about 6.6 times as likely to be LBW (49.4% vs 7.5%) but there are only 79 of them, so they account for 39 of the 356 LBW events (11%). `multiple` is a very strong but narrow signal.
- 18 later pregnancies had a missing interval; they are set to the cohort median (39 months) and flagged in `interval_imputed`.

## 8. Distributions by outcome

Unweighted histograms, scaled by group share, of a few continuous
predictors. These are shape checks; the weighted rates by group follow.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, bins in zip(
    axes, ["maternal_age", "anc_visits", "first_anc_month"], [30, 20, 11]
):
    for v, label in [(0, "Not LBW"), (1, "LBW")]:
        ax.hist(df.loc[df[OUTCOME] == v, col].dropna(),
                bins=bins, density=True, alpha=0.5, label=label)
    ax.set_title(col)
    ax.set_ylabel("Share of group")
axes[0].legend()
plt.suptitle("Kenya LBW cohort: distributions by outcome (unweighted)")
plt.tight_layout()
plt.show()

In [ ]:
hist_cols = ["maternal_age", "anc_visits", "first_anc_month"]
display(df.groupby(OUTCOME)[hist_cols].agg(["mean", "median"]).round(2))
print("Share aged <20:", df.groupby(OUTCOME)["maternal_age"].apply(lambda s: (s < 20).mean()).round(3).to_dict())
print("Birth weights exactly on the cut-off (2500 g):", int((df["birth_weight_g"] == 2500).sum()))
print("Birth weights recorded as multiples of 100 g:", f"{(df['birth_weight_g'] % 100 == 0).mean():.0%}")

**Result:**
- Mothers of LBW babies are slightly younger (mean 25.8 vs 26.6 years; median 24 vs 26). 16.3% are under 20, against 12.7% for non-LBW.
- ANC visits (mean 4.2 vs 4.3) and month of first ANC visit (mean 4.3 vs 4.3) are almost identical between the groups, so neither separates LBW at the distribution level.

These are unweighted shapes; the weighted group rates follow.

## 9. LBW by background factor and antenatal care

Weighted LBW rate per 1,000 in each group, with a 95% CI and a `few_events`
flag for cells with fewer than 30 events.

In [ ]:
BASE_GROUPS = {
    "age_group":      ["<18", "18-24", "25-34", "35+"],
    "parity_group":   ["0", "1-2", "3-4", "5+"],
    "interval_group": ["First pregnancy", "<24 months", "24-35 months", "36+ months"],
    "prior_stillbirth": [0, 1],
    "prior_loss":     [0, 1],
    "multiple":       [0, 1],
    "education":      ["No education", "Primary", "Secondary", "Higher"],
    "wealth_quintile": ["Poorest", "Poorer", "Middle", "Richer", "Richest"],
    "residence":      ["Urban", "Rural"],
}
base_tables = {v: rate_by(df, v, order=o) for v, o in BASE_GROUPS.items()}
display(pd.concat(base_tables, names=["variable", "level"]))
plot_rates(base_tables, "Kenya LBW cohort: weighted LBW rate by background factor")

**Result (weighted LBW per 1,000; CIs ignore clustering, so treat borderline gaps with caution; overall 78.4):**
- **Multiple birth:** 435 (twins) vs 72 (singletons). By far the largest gap.
- **Prior loss:** 122 vs 72; the CIs barely overlap (89-165 vs 62-85). **Prior stillbirth:** 103 vs 78, but only 111 women and 10 events (`few_events`).
- **Education:** gradient from 109 (none) and 95 (primary) to 74 (secondary) and 52 (higher).
- **Wealth:** not monotonic. The richest quintile is lowest (48; CI 30-77) while the poorer quintile is highest (107); the other three sit between 77 and 87.
- **Residence:** rural 88 vs urban 64.
- **Age:** under 18 is highest (105, but only 19 events), 18-24 is 94, 25-34 is lowest (64) and 35+ is 76 with a wide CI (51-111). The elevation is in younger mothers; the 35+ excess is weak.
- **Parity:** a mild U-shape (0: 84, 1-2: 78, 3-4: 65, 5+: 93) with heavily overlapping CIs.
- **Interval:** <24 months is highest (93) but overlaps the rest (24-35: 79, 36+: 74). First pregnancies (76) are *not* elevated relative to 36+ months.
- Education, wealth and residence directions should be read with the facility-birth caveat and are not causal.

In [ ]:
coverage = pd.Series({
    "No ANC at all":              weighted_rate(df, "no_anc", per=100),
    "4+ visits":                  weighted_rate(df, "anc4plus", per=100),
    "First visit in 1st trimester":
        100 * np.average(df["first_anc_month"].between(1, 3), weights=df["weight"]),
    "BP measured":
        weighted_rate(df.dropna(subset=["bp_measured"]), "bp_measured", per=100),
    "Urine tested":
        weighted_rate(df.dropna(subset=["urine_tested"]), "urine_tested", per=100),
    "Blood tested":
        weighted_rate(df.dropna(subset=["blood_tested"]), "blood_tested", per=100),
    "Iron tablets":
        weighted_rate(df.dropna(subset=["iron_tablets"]), "iron_tablets", per=100),
}, name="weighted %")
display(coverage.to_frame())

In [ ]:
ANC_GROUPS = {
    "anc_group":           ["None", "1-3", "4-7", "8+"],
    "first_anc_trimester": ["No ANC", "1st (0-3 m)", "2nd (4-6 m)", "3rd (7+ m)"],
    "bp_measured":   [0, 1],
    "urine_tested":  [0, 1],
    "blood_tested":  [0, 1],
    "iron_tablets":  [0, 1],
}
anc_tables = {v: rate_by(df, v, order=o) for v, o in ANC_GROUPS.items()}
display(pd.concat(anc_tables, names=["variable", "level"]))
plot_rates(anc_tables, "Kenya LBW cohort: weighted LBW rate by antenatal-care factor")

**Result:**
- Coverage (weighted): no ANC 0.7%; 4+ visits 70.3%; first visit in the first trimester 30.2%; **BP measured 97.9%, urine tested 96.7%** (the two hypertension-detecting checks); blood tested 97.3%; iron tablets 93.8%.
- ANC effects on LBW are weak and hard to read, as expected for an outcome driven by twins and constitutional factors rather than by ANC attendance:
  - Visits: 4-7 visits is lowest (65 per 1,000), 1-3 is 93, and **8+ visits is highest (184; 22 events, CI 112-287)**. More visits likely reflect higher-risk pregnancies or referral (reverse causation), not harm.
  - Timing: first-trimester booking (89) is higher than second (74) and third (65), the same reverse-causation pattern.
  - BP measured (78 vs 83) and urine tested (78 vs 88) show no protective effect, and the "not tested" groups have 15-20 events, so these comparisons are imprecise.
  - Iron tablets: 76 vs 122 for women without them (CIs 65-87 vs 76-191); this is the largest gap among the yes/no ANC items but still imprecise, and women with short pregnancies have less time to receive them.

# 10. Leakage check — ANC visits reflect **pregnancy length**

A 7-month pregnancy has had less time for ANC visits than a 9-month one,
and early-ending pregnancies are also where LBW concentrates. If a model
sees raw `anc_visits` it partly uses "pregnancy ended early", which is only
known at the end. This is **more severe for LBW than for perinatal death**,
because preterm birth is a dominant cause of LBW.

`pregnancy_months` is used here **only to check**; it never enters a model.

In [ ]:
trunc = df.groupby("pregnancy_months").agg(
    n=("anc_visits", "size"),
    mean_anc_visits=("anc_visits", "mean"),
    lbw=(OUTCOME, "sum"),
)
trunc["lbw_rate_per_1000"] = 1000 * trunc["lbw"] / trunc["n"]
display(trunc)

# Visual
fig, ax = plt.subplots(figsize=(7, 3.5))
trunc["lbw_rate_per_1000"].plot(kind="bar", ax=ax, color="steelblue")
ax.set_ylabel("LBW per 1,000")
ax.set_xlabel("Pregnancy length (months)")
ax.set_title("LBW rate by pregnancy length (leakage check); unweighted")
plt.show()

print("Share of all LBW events that occur at 9+ months:",
      f"{df.loc[df['pregnancy_months'] >= 9, OUTCOME].sum() / df[OUTCOME].sum():.1%}")
print("Women at 9 months:", int((df['pregnancy_months'] == 9).sum()),
      "| LBW events at 9 months:", int(df.loc[df['pregnancy_months'] == 9, OUTCOME].sum()))

**Result:**
- LBW falls from **655 per 1,000 at 7 months** (58 women) to 209 at 8 months (254), **67 at 9 months** (3,892) and 42 at 10 months (95): a tenfold drop from 7 to 9 months.
- Mean ANC visits rise with pregnancy length (3.78, 4.19, 4.29, 5.38 for 7, 8, 9, 10 months). The 7-to-9-month difference is only about half a visit, so the leak through raw `anc_visits` is real but modest in size; it is clearest at 10 months.
- 74.4% of LBW events (265 of 356) occur at 9+ months, so most LBW in this cohort is not early delivery. This is why the rank correlation with `pregnancy_months` is only -0.20 (§14), yet it remains a post-birth variable.
- The 9-month stratum has 3,892 women and 261 events, enough for the sensitivity analysis.

**Recommendation to carry into the modelling notebooks:**

## 11. Hypertension angle (exploratory)

Hypertension in pregnancy is central to our qualitative work but not
recorded directly in DHS. Two proxies: BP measured at ANC (delivery-side)
and "ever told high BP" (asked at interview; timing uncertain).
Exploratory only.

In [ ]:
print("told_high_bp answered by:",
      f"{df['told_high_bp'].notna().mean():.0%}", "of women")

htn = pd.concat({
    "told_high_bp": rate_by(df, "told_high_bp", order=[0, 1]),
    "bp_measured":  rate_by(df.dropna(subset=["bp_measured"]), "bp_measured", order=[0, 1]),
}, names=["variable", "level"])
display(htn)

# Is told_high_bp mostly among those whose BP was measured?
display(pd.crosstab(df["bp_measured"], df["told_high_bp"], margins=True))

**Result:**
1. **`told_high_bp` and LBW.** 363 women (8.4%) report being told they had high BP. Their LBW rate is 98.9 per 1,000 (CI 64-149; 45 events) against 76.3 (CI 66-88) for the rest. The point estimate is higher but the CIs overlap, and they ignore clustering, so the association is not established. The `= 1` cell has 45 events, so `few_events` does not fire. `bp_measured` shows no difference (78 measured vs 83 unmeasured), and the unmeasured group has only 115 women and 15 events.
2. **Crosstab.** Of 4,180 women with BP measured at ANC, 348 (8.3%) report being told high BP; of the 115 unmeasured, 15 (13.0%) do. So being "told" is not confined to women measured at ANC; those 15 were diagnosed elsewhere or earlier. The data cannot identify undiagnosed hypertension, only diagnosed hypertension (4 women have `bp_measured` missing).

`told_high_bp` stays **exploratory only**; never a model feature, because the diagnosis timing is uncertain relative to this pregnancy.

## 12. Region (exploratory only)

Kenyan counties and Tanzanian regions are different units, so `region`
cannot enter a pooled model. Rates are also unstable because most counties
have fewer than 30 LBW events. Use for report context only.

In [ ]:
region = rate_by(df, "region").sort_values("rate_per_1000", ascending=False)
display(region.head(10))
print("Counties with < 30 LBW events:",
      int(region["few_events"].sum()),
      "of", len(region))

**Result:** all 47 counties have fewer than 30 LBW events (the most is 19, in Isiolo). The highest rates (Wajir 292 per 1,000 on 15 events of 51 births; Isiolo 191; Laikipia 176) all have very wide CIs (Wajir 169-457), so counties should not be ranked against each other. Report as context only; region is excluded from the pooled model because the units differ.

## 13. Checklist baseline

The current risk checklist was designed for perinatal death, not LBW. Here
we report how it performs on LBW, so the modelling notebooks have a stated
target.

In [ ]:
display(
    screening_summary(df["checklist_flag"], df[OUTCOME])
    .to_frame("checklist (Kenya LBW cohort)")
)

print()
print(pd.crosstab(df["checklist_flag"], df[OUTCOME], rownames=["flagged"], colnames=["LBW"], margins=True))
print(f"Base rate (unweighted): {100 * df[OUTCOME].mean():.1f}%")

**Result (unweighted):** the checklist flags **36.4%** of pregnancies (1,564) and catches 166 of 356 LBW births.
- Sensitivity 46.6%; specificity 64.5%.
- PPV 10.6% (against a base rate of 8.3%, so a flagged woman is only about 1.3 times as likely to have an LBW baby); NPV 93.1%.
- 190 LBW births (53%) are not flagged.

Target for the modelling notebooks: **"the checklist baseline for LBW in this cohort is sensitivity 46.6%, PPV 10.6% at 36.4% flagged. Our model must beat this at a comparable flagged fraction."**

## 14. Correlations between predictors

Strongly correlated predictors carry the same information twice, which
makes logistic regression coefficients unstable and SHAP explanations
confusing. Spearman is used because several predictors are counts / flags.
Include `birth_weight_g` and `pregnancy_months` in this heatmap as a sanity
check (leakage signal), not as features.

In [ ]:
corr_cols = [
    "maternal_age", "prior_pregnancies", "prior_live_births",
    "prior_loss", "prior_stillbirth", "first_pregnancy", "short_interval",
    "multiple", "anc_visits", "first_anc_month", "bp_measured",
    "urine_tested", "blood_tested", "iron_tablets",
    "pregnancy_months", "birth_weight_g",
    OUTCOME,
]
corr = df[corr_cols].corr(method="spearman")

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)), corr_cols, rotation=90)
ax.set_yticks(range(len(corr_cols)), corr_cols)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}",
                ha="center", va="center", fontsize=7)
fig.colorbar(im)
ax.set_title("Spearman correlations, Kenya LBW cohort")
plt.tight_layout()
plt.show()

high = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
display(high[high.abs() > 0.7].sort_values(ascending=False))

print("\nSpearman correlation with the outcome:")
print(corr[OUTCOME].drop(OUTCOME).sort_values().round(2).to_string())

**Result:** pairs with |rho| > 0.7:
- `prior_pregnancies` and `prior_live_births` (0.97): keep `prior_live_births` (matches the checklist), **drop `prior_pregnancies`** from model inputs.
- `maternal_age` with `prior_pregnancies` (0.72) and `prior_live_births` (0.71): fine for tree models; watch for unstable coefficients in logistic regression.
- `first_pregnancy` with `prior_pregnancies` (-0.79) and `prior_live_births` (-0.77): a construction artefact, since `first_pregnancy` is derived from them. Fine.

Correlation with the outcome: `birth_weight_g` -0.48 (the label definition, not a discovery), `pregnancy_months` -0.20 (the leak, quantified; as large as the strongest legitimate predictor, `multiple` at +0.20), `prior_loss` +0.05, and every other variable |rho| <= 0.05.

No ANC variable is correlated above 0.7 with any other predictor.

## 15. Summary: what this means for modelling

### Data quality
- **4,299 pregnancies, 356 LBW events** (8.3% unweighted, 7.8% weighted): matches `cohort_config.yaml`. One row per woman, no duplicates, all Kenya.
- Missingness: `preceding_interval_m` 27.8% (all 1,179 first pregnancies plus 18 later ones); six ANC fields have at most 4 blanks each; everything else is complete. **Decision:** use `first_pregnancy` + `interval_group`; the 18 later pregnancies with unknown interval are set to the cohort median of **39 months** in `build_cohort()` and flagged `interval_imputed`. Recompute the median on the training split in the models.
- **No stillbirths in the file:** every row is "Born alive" with `stillbirth == 0`.
- **Only weighed babies are included, 97.3% from facility births.** Results generalise to the facility population, not to Kenya as a whole.

### Outcome
- Weighted LBW rate **78.4 per 1,000** (unweighted 82.8). Around 8% is imbalanced, so use PR-AUC (not accuracy) and class weights.

### Strongest signals
- **`multiple`:** twins 435 per 1,000 vs 72 for singletons, by far the largest effect, but only 79 twin births (11% of LBW events).
- **`prior_loss`** (122 vs 72) and `prior_stillbirth` (103 vs 78, few events).
- **Education** (109 none to 52 higher), **residence** (rural 88 vs urban 64) and **wealth** (richest lowest at 48, otherwise non-monotonic).
- **Age:** under 18 (105) and 18-24 (94) are elevated; 25-34 is lowest (64); the 35+ excess is weak (76).
- **Parity** (mild U-shape) and **birth interval** (<24 months 93) show smaller differences with overlapping CIs; first pregnancies are not elevated.

### Interpret with care
- Wealth / residence / education: record the direction, apply the facility-birth caveat, do not read as causal.
- ANC is weak and confusing: 8+ visits (184) and first-trimester booking (89) show higher LBW, consistent with reverse causation. BP measured and urine tested show no protective effect.
- ANC delivery metrics: BP measured 97.9%, urine tested 96.7%.

### Leakage (most important for LBW)
- **`pregnancy_months`** is the clearest leak (655 per 1,000 at 7 months vs 67 at 9; rho = -0.20 with LBW). **`anc_visits`** rises with pregnancy length (3.8 at 7 months to 5.4 at 10).
- Recommendation: **drop raw `anc_visits`**; use `anc4plus` and `first_anc_month`; run a sensitivity analysis on `pregnancy_months == 9` (3,892 women, 261 events).

### Hypertension (exploratory)
- `told_high_bp` is complete in this file. 8.4% report being told; LBW 98.9 vs 76.3 per 1,000 with overlapping CIs, so not established.
- Only diagnosed (not undiagnosed) hypertension can be inferred. Keep exploratory only.

### Checklist baseline
- Flags 36.4%; sensitivity 46.6%, specificity 64.5%, PPV 10.6%, NPV 93.1% (base rate 8.3%). This is the number the models must beat.

### Correlations
- Drop `prior_pregnancies` from model inputs (0.97 with `prior_live_births`).
- Maternal age / parity correlation (0.71-0.72): trees are fine; logit coefficients may be unstable.
- No ANC variable is correlated above 0.7 with another predictor.

### Hand-off to modelling notebooks
- `build_cohort("lbw")` produces the same `interval_group`, `first_pregnancy`, `short_interval`, `interval_imputed`, `no_anc`, `anc4plus`, `checklist_flag` defined here.
- `SELECTED` in `dataset.py` should include at least: `low_birth_weight`, `split`, `cluster_id`, `country`, `weight`, plus the base and ANC predictors above.